In [1]:
# 对token以及embedding过程的探究

In [2]:
import sys
from transformers import GPT2LMHeadModel, GPT2Tokenizer, PreTrainedModel
from transformers.modeling_outputs import CausalLMOutputWithCrossAttentions
from loguru import logger
import torch
import torch.nn.functional as F
import os
from typing import cast

os.environ["HTTP_PROXY"] = "http://127.0.0.1:6382"
os.environ["HTTPS_PROXY"] = "http://127.0.0.1:6382"

logger.remove()
logger.add(sys.stdout, level="DEBUG", colorize=True)

gpt2_tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
gpt2_model = GPT2LMHeadModel.from_pretrained("gpt2")
gpt2_model.eval();

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

首先构造基本的类，用于支持后续的观察与试验，本类用于展示token之间以及embedding vectors之间的关系。

In [3]:
class TokenAndEmbedding:
    """
    用于展示token及其对应的embedding的类，并提供了其他探究embedding vector的方法
    """

    def __init__(self, model: PreTrainedModel, tokenizer: GPT2Tokenizer) -> None:
        self._model: PreTrainedModel = model
        self._tokenizer: GPT2Tokenizer = tokenizer
        self._embedding_table = model.transformer.wte.weight.detach()

    def display_word_similarity(self, prompt: str) -> None:
        """
        使用方向余弦来展示提示词中各单词的关联度
        :param prompt: 提示词字符串
        :return: None
        """

        # 获取token列表以及其对应的embedding_vectors
        token_ids: list[int] = self._tokenizer.encode(prompt)
        tokens: list[str] = [cast(str, self._tokenizer.decode([token_id])) for token_id in token_ids]
        embedding_vectors: dict[str, torch.Tensor] = self._get_prompt_embedding_vectors(prompt)

        # 统计不同词对之间的相似关系，这里两个词之间的相似关系只统计一次，且不统计自己与自己的相似性
        computed_pairs: set[tuple[str, ...]] = set()
        similarity_results: list[tuple[str, str, float]] = []
        for token1 in tokens:
            for token2 in tokens:
                if token1 >= token2:
                    continue
                key = tuple(sorted([token1, token2]))
                if key in computed_pairs:
                    continue
                computed_pairs.add(key)
                if token1 in embedding_vectors and token2 in embedding_vectors:
                    sim = cast(float,
                               F.cosine_similarity(embedding_vectors[token1], embedding_vectors[token2], dim=0).item())
                    similarity_results.append((token1, token2, sim))

        # 展示各token相似度情况
        similarity_results.sort(key=lambda x: -x[2])
        for w1, w2, sim in similarity_results:
            bar = "█" * int(max(0, sim) * 30)
            logger.info(f"  '{w1}' vs '{w2}': {sim:>7.3f}  {bar}")

    def display_analogy(self, a: str, b: str, c: str, top_k: int = 8) -> list[tuple[str, float]]:
        """
        计算：向量(a) - 向量(b) + 向量(c)
        """
        vec_a, id_a, repr_a = self._get_embedding_vector(a)
        vec_b, id_b, repr_b = self._get_embedding_vector(b)
        vec_c, id_c, repr_c = self._get_embedding_vector(c)
        target: torch.Tensor = vec_a - vec_b + vec_c
        logger.info(f"公式：向量('{a}') - 向量('{b}') + 向量('{c}') = ")

        exclude_ids: list[int] = [id_a, id_b, id_c]
        for word in (a, b, c):
            exclude_ids.extend(token_id for _, token_id in self._get_word_variants_with_ids(word))

        results = self._find_nearest_tokens(target, exclude_ids=exclude_ids, top_k=top_k)
        logger.info(f"排除输入词后，最接近的{top_k}个词：")
        for i, (token, sim) in enumerate(results):
            bar = "█" * int(max(0, sim) * 25)
            marker = "✅❓" if i == 0 else ""
            logger.info(f"{i + 1}.'{token!r}'的相似度{sim:.3f} {bar}{marker}")
        return results

    def _find_nearest_tokens(self, target_vector: torch.Tensor, exclude_ids: list[int] | None = None, top_k: int = 8) -> \
            list[tuple[str, float]]:
        """在词表中找到最接近 target_vector 的 token"""
        if exclude_ids is None:
            exclude_ids = list()

        # 计算方向余弦，词表中的每一个词都进行计算
        norms: torch.Tensor = self._embedding_table.norm(dim=1)
        target_norm: torch.Tensor = target_vector.norm()
        similarities = torch.mv(self._embedding_table, target_vector) / (norms * target_norm + 1e-8)

        for exclude_id in exclude_ids:
            similarities[exclude_id] = -float("inf")

        top_values, top_indices = torch.topk(similarities, top_k)
        results: list[tuple[str, float]] = []
        for i in range(top_k):
            token: str = cast(str, self._tokenizer.decode([top_indices[i].item()]))
            sim = top_values[i].item()
            results.append((token, sim))
        return results

    def _get_prompt_embedding_vectors(self, prompt: str) -> dict[str, torch.Tensor]:
        """
        预览提示词embedding vectors，了解其embedding的形式
        :param prompt: 提示词字符串
        :return: token以及其embedding vectors的字典
        """
        token_ids: list[int] = self._tokenizer.encode(prompt)
        tokens: list[str] = [cast(str, self._tokenizer.decode([token_id])) for token_id in token_ids]
        vectors: dict[str, torch.Tensor] = {}
        for token in tokens:
            vec, tid, display = self._get_embedding_vector(token)
            vectors[token] = vec
            preview = [f"{v:.3f}" for v in vec[:8].tolist()]
            logger.info(f"word = '{display}', ID = {tid}, vec = [{', '.join(preview)}, …]")
        return vectors

    def _get_embedding_vector(self, word: str) -> tuple[torch.Tensor, int, str]:
        """
        获取一个词的Embedding向量，自动处理带/不带空格的变体，如无法匹配单个token，则返回第一个子token信息
        """
        variants_with_ids = self._get_word_variants_with_ids(word)
        if variants_with_ids:
            variant, token_id = variants_with_ids[0]
            return self._embedding_table[token_id], token_id, variant.strip(),

        token_ids: list[int] = self._tokenizer.encode(word)
        token_id = token_ids[0]
        token = cast(str, self._tokenizer.decode([token_id]))
        return self._embedding_table[token_id], token_id, token.strip()

    def _get_word_variants_with_ids(self, word: str) -> list[tuple[str, int]]:
        """获取一个词所有可编码为单 token 的常见变体及其 ID。"""
        base_word = word.strip()
        variants = [word, base_word, f" {base_word}", base_word.lower(), f" {base_word.lower()}",
                    base_word.capitalize(), f" {base_word.capitalize()}"]
        results: list[tuple[str, int]] = []
        seen_ids: set[int] = set()

        for variant in variants:
            token_ids: list[int] = self._tokenizer.encode(variant)
            if len(token_ids) == 1:
                token_id = token_ids[0]
                if token_id not in seen_ids:
                    results.append((variant, token_id))
                    seen_ids.add(token_id)
        return results

首先测试展示一些词语之间的相关关系，使用方向余弦可以看出不同词之间的关联性，同时也可以看出不同词语所对应的embedding vector。

In [4]:
token_and_embedding = TokenAndEmbedding(gpt2_model, gpt2_tokenizer)
token_and_embedding.display_word_similarity("France Paris Germany Berlin cat dog the king queen man woman")

2026-08-07 23:56:11.111 | INFO     | __main__:_get_prompt_embedding_vectors:102 - word = 'France', ID = 28572, vec = [-0.133, 0.098, 0.230, -0.105, -0.011, 0.047, -0.217, -0.021, …]
2026-08-07 23:56:11.112 | INFO     | __main__:_get_prompt_embedding_vectors:102 - word = 'Paris', ID = 6342, vec = [-0.219, 0.042, 0.121, -0.260, 0.028, -0.051, -0.229, -0.022, …]
2026-08-07 23:56:11.113 | INFO     | __main__:_get_prompt_embedding_vectors:102 - word = 'Germany', ID = 4486, vec = [-0.131, 0.046, 0.041, 0.077, 0.093, 0.031, -0.224, -0.005, …]
2026-08-07 23:56:11.113 | INFO     | __main__:_get_prompt_embedding_vectors:102 - word = 'Berlin', ID = 11307, vec = [-0.247, 0.041, 0.010, -0.134, -0.048, -0.054, -0.228, -0.100, …]
2026-08-07 23:56:11.113 | INFO     | __main__:_get_prompt_embedding_vectors:102 - word = 'cat', ID = 3797, vec = [0.010, 0.037, 0.164, -0.219, 0.029, -0.098, -0.336, -0.075, …]
2026-08-07 23:56:11.114 | INFO     | __main__:_get_prompt_embedding_vectors:102 - word = 'dog', ID

再展示不同词语的“运算”关系，包含很经典的 king - man + woman = queen。

In [5]:
token_and_embedding.display_analogy(" king", " man", " woman")

2026-08-07 23:56:11.173 | INFO     | __main__:display_analogy:53 - 公式：向量(' king') - 向量(' man') + 向量(' woman') = 
2026-08-07 23:56:11.181 | INFO     | __main__:display_analogy:60 - 排除输入词后，最接近的8个词：
2026-08-07 23:56:11.182 | INFO     | __main__:display_analogy:64 - 1.'' queen''的相似度0.709 █████████████████✅❓
2026-08-07 23:56:11.182 | INFO     | __main__:display_analogy:64 - 2.'' princess''的相似度0.605 ███████████████
2026-08-07 23:56:11.182 | INFO     | __main__:display_analogy:64 - 3.'' Queen''的相似度0.596 ██████████████
2026-08-07 23:56:11.182 | INFO     | __main__:display_analogy:64 - 4.'' kings''的相似度0.593 ██████████████
2026-08-07 23:56:11.182 | INFO     | __main__:display_analogy:64 - 5.''Queen''的相似度0.572 ██████████████
2026-08-07 23:56:11.183 | INFO     | __main__:display_analogy:64 - 6.'' monarch''的相似度0.525 █████████████
2026-08-07 23:56:11.183 | INFO     | __main__:display_analogy:64 - 7.'' queens''的相似度0.519 ████████████
2026-08-07 23:56:11.183 | INFO     | __main__:display_analogy:64 - 8

[(' queen', 0.7085265517234802),
 (' princess', 0.6046253442764282),
 (' Queen', 0.5964063405990601),
 (' kings', 0.59321528673172),
 ('Queen', 0.5719681978225708),
 (' monarch', 0.5248165726661682),
 (' queens', 0.5192992687225342),
 (' goddess', 0.5166212320327759)]

In [6]:
token_and_embedding.display_analogy("Paris", "France", "Germany")

2026-08-07 23:56:11.197 | INFO     | __main__:display_analogy:53 - 公式：向量('Paris') - 向量('France') + 向量('Germany') = 
2026-08-07 23:56:11.202 | INFO     | __main__:display_analogy:60 - 排除输入词后，最接近的8个词：
2026-08-07 23:56:11.202 | INFO     | __main__:display_analogy:64 - 1.'' Berlin''的相似度0.587 ██████████████✅❓
2026-08-07 23:56:11.203 | INFO     | __main__:display_analogy:64 - 2.''German''的相似度0.561 ██████████████
2026-08-07 23:56:11.203 | INFO     | __main__:display_analogy:64 - 3.'' Munich''的相似度0.557 █████████████
2026-08-07 23:56:11.203 | INFO     | __main__:display_analogy:64 - 4.''London''的相似度0.530 █████████████
2026-08-07 23:56:11.203 | INFO     | __main__:display_analogy:64 - 5.'' Bayern''的相似度0.524 █████████████
2026-08-07 23:56:11.203 | INFO     | __main__:display_analogy:64 - 6.'' Hamburg''的相似度0.522 █████████████
2026-08-07 23:56:11.203 | INFO     | __main__:display_analogy:64 - 7.''Moscow''的相似度0.519 ████████████
2026-08-07 23:56:11.203 | INFO     | __main__:display_analogy:64 - 8.'' 

[(' Berlin', 0.5870515704154968),
 ('German', 0.5607730746269226),
 (' Munich', 0.5571812391281128),
 ('London', 0.530412495136261),
 (' Bayern', 0.5236919522285461),
 (' Hamburg', 0.5222302079200745),
 ('Moscow', 0.5192763209342957),
 (' Cologne', 0.5162988901138306)]

除了展示token以及其与embedding vectors之外，还需要探究同一个词在不同上下文，或者含有不同语意时的情况。

首先构造基本的类，用于支持后续的观察与试验，本类用于展示同一个token在同义词情况下，在transformer过程中余弦相似度逐渐拉开的过程，同时展示位置对token上下文信息的影响。

In [7]:
class TokenInContext:
    """
    用于展示位于不同位置的token如何反应不同上下文信息的影响
    """

    def __init__(self, model: PreTrainedModel, tokenizer: GPT2Tokenizer) -> None:
        self._model: PreTrainedModel = model
        self._tokenizer: GPT2Tokenizer = tokenizer
        self._embedding_table = model.transformer.wte.weight.detach()

    def display_polysemy_word(self) -> None:
        """展示多义词 bank 在不同上下文中的向量如何逐层分化。"""
        sentences: list[tuple[str, str, str]] = [
            ("I put money in the bank yesterday", "bank", "银行"),
            ("I saw fish near the bank quietly", "bank", "河岸"),
            ("I got cash from the bank quickly", "bank", "银行"),
            ("I saw water near the bank outside", "bank", "河岸")
        ]

        all_layer_vectors: list[list[torch.Tensor]] = []

        for text, target_word, meaning in sentences:
            layer_vectors, tokens, target_pos = self._get_all_layer_vectors(text, target_word)

            if layer_vectors is None or target_pos is None:
                raise RuntimeError(f"未找到目标词：{target_word!r}")

            all_layer_vectors.append(layer_vectors)

            logger.info(f"{meaning}：{text!r}，"
                        f"目标 token={tokens[target_pos]!r}，位置={target_pos}")

        finance_1 = all_layer_vectors[0]
        river_1 = all_layer_vectors[1]
        finance_2 = all_layer_vectors[2]
        river_2 = all_layer_vectors[3]

        layer_count = len(finance_1)

        logger.info("逐层比较 bank 的上下文向量：")

        for layer_index in range(layer_count):
            finance_similarity = F.cosine_similarity(finance_1[layer_index], finance_2[layer_index], dim=0).item()
            river_similarity = F.cosine_similarity(river_1[layer_index], river_2[layer_index], dim=0).item()
            cross_similarities: list[float] = [
                F.cosine_similarity(finance_1[layer_index], river_1[layer_index], dim=0).item(),
                F.cosine_similarity(finance_1[layer_index], river_2[layer_index], dim=0).item(),
                F.cosine_similarity(finance_2[layer_index], river_1[layer_index], dim=0).item(),
                F.cosine_similarity(finance_2[layer_index], river_2[layer_index], dim=0).item(),
            ]

            same_mean = (finance_similarity + river_similarity) / 2
            cross_mean = sum(cross_similarities) / len(cross_similarities)
            separation = same_mean - cross_mean

            layer_name = ("Embedding" if layer_index == 0 else f"Transformer {layer_index}")

            logger.info(f"{layer_name:>14}: 银行内部={finance_similarity:.3f}, 河岸内部={river_similarity:.3f}, "
                        f"同义平均={same_mean:.3f}, 异义平均={cross_mean:.3f}, 分离度={separation:+.3f}")

    def display_word_position_embedding(self) -> None:
        sentences: list[str] = [
            "The man bites dog",
            "The dog bites man"
        ]
        for sentence in sentences:
            input_ids: list[int] = self._tokenizer.encode(sentence)
            token_ids: torch.Tensor = torch.tensor([input_ids], dtype=torch.long)
            tokens: list[str] = [cast(str, self._tokenizer.decode(token_id)) for token_id in input_ids]
            logger.info(f"sentence: {sentence!r}")

            with torch.no_grad():
                outputs = cast(CausalLMOutputWithCrossAttentions, self._model(token_ids, output_hidden_states=True))
            hidden_states = outputs.hidden_states
            if hidden_states is None:
                raise RuntimeError("hidden_states is None")
            final_vectors = hidden_states[-1][0]

            for position, (token_id, token) in enumerate(zip(token_ids[0], tokens)):
                wte_vector = self._model.transformer.wte.weight[token_id].detach()
                wpe_vector = self._model.transformer.wpe.weight[position].detach()
                input_vector = wte_vector + wpe_vector
                logger.info(f"position = {position}, token = {token!r}")
                logger.info(f"input_vector[:3]: {input_vector[:3].tolist()}")
                logger.info(f"wte_vector[:3]: {wte_vector[:3].tolist()}")
                logger.info(f"wpe_vector[:3]: {wpe_vector[:3].tolist()}")

    def _get_all_layer_vectors(self, text: str, target_word: str) -> tuple[
        list[torch.Tensor] | None, list[str], int | None]:
        """
        获取某一个词的所有层对应的向量
        :param text: prompt
        :param target_word: 目标词
        :return: 元组，包含目标向量、token列表、目标位置
        """
        token_ids: list[int] = self._tokenizer.encode(text)
        input_ids: torch.Tensor = torch.tensor([token_ids], dtype=torch.long)
        tokens: list[str] = [cast(str, self._tokenizer.decode(token_id)) for token_id in input_ids[0]]

        # 目标词的位置
        target_pos: int | None = None
        for i, t in enumerate(tokens):
            if t.lower().strip() == target_word.lower():
                target_pos = i
                break

        if target_pos is None:
            return None, tokens, None

        # # hidden_states[0] = Embedding 层, [1]-[12] = Transformer 各层
        with torch.no_grad():
            outputs = cast(CausalLMOutputWithCrossAttentions, self._model(input_ids, output_hidden_states=True))
        hidden_states = outputs.hidden_states
        if hidden_states is None:
            raise RuntimeError("hidden_states is None")
        layer_vectors = [hidden_state[0, target_pos, :].detach() for hidden_state in hidden_states]
        return layer_vectors, tokens, target_pos

首先展示bank的不同含义在transformer过程中的演化。从结果中可以看到，在transformer逐步加深的过程中，二者含义逐渐分离，但是最后又趋于类似，同一 token 的表示会在 Transformer 中逐层融入上下文语义并逐渐分化，但最终输出表示经过最后一层及 ln_f 后，其余弦相似度会再次升高，因此语义分离并不随层数单调增强。

In [8]:
token_in_context = TokenInContext(gpt2_model, gpt2_tokenizer)
token_in_context.display_polysemy_word()

2026-08-07 23:56:11.264 | INFO     | __main__:display_polysemy_word:30 - 银行：'I put money in the bank yesterday'，目标 token=' bank'，位置=5
2026-08-07 23:56:11.278 | INFO     | __main__:display_polysemy_word:30 - 河岸：'I saw fish near the bank quietly'，目标 token=' bank'，位置=5
2026-08-07 23:56:11.294 | INFO     | __main__:display_polysemy_word:30 - 银行：'I got cash from the bank quickly'，目标 token=' bank'，位置=5
2026-08-07 23:56:11.310 | INFO     | __main__:display_polysemy_word:30 - 河岸：'I saw water near the bank outside'，目标 token=' bank'，位置=5
2026-08-07 23:56:11.310 | INFO     | __main__:display_polysemy_word:40 - 逐层比较 bank 的上下文向量：
2026-08-07 23:56:11.311 | INFO     | __main__:display_polysemy_word:58 -      Embedding: 银行内部=1.000, 河岸内部=1.000, 同义平均=1.000, 异义平均=1.000, 分离度=+0.000
2026-08-07 23:56:11.311 | INFO     | __main__:display_polysemy_word:58 -  Transformer 1: 银行内部=0.996, 河岸内部=0.994, 同义平均=0.995, 异义平均=0.983, 分离度=+0.011
2026-08-07 23:56:11.311 | INFO     | __main__:display_polysemy_word:58 -  Trans

接下来探究位置对token对应vector的影响。实际上送入transformer的向量包含了WTE(word token embedding)和WPE(word position embedding)两部分。对于GPT-2来说，任意两句话的第i个token的WPE是完全一致的。GPT-2 的 WPE 是一张可训练的位置向量表，每个绝对位置对应一个固定的 768 维参数向量；训练时它与 WTE、Transformer 参数一起通过 next-token prediction 的 loss 进行反向传播更新，因此同一模型中所有句子的相同 position 都使用完全相同的 WPE。

In [9]:
token_in_context.display_word_position_embedding()

2026-08-07 23:56:11.375 | INFO     | __main__:display_word_position_embedding:70 - sentence: 'The man bites dog'
2026-08-07 23:56:11.391 | INFO     | __main__:display_word_position_embedding:83 - position = 0, token = 'The'
2026-08-07 23:56:11.392 | INFO     | __main__:display_word_position_embedding:84 - input_vector[:3]: [-0.08744361251592636, -0.21771633625030518, 0.06849727779626846]
2026-08-07 23:56:11.392 | INFO     | __main__:display_word_position_embedding:85 - wte_vector[:3]: [-0.0686228945851326, -0.020297741517424583, 0.0644705519080162]
2026-08-07 23:56:11.392 | INFO     | __main__:display_word_position_embedding:86 - wpe_vector[:3]: [-0.01882071979343891, -0.19741860032081604, 0.004026724956929684]
2026-08-07 23:56:11.392 | INFO     | __main__:display_word_position_embedding:83 - position = 1, token = ' man'
2026-08-07 23:56:11.393 | INFO     | __main__:display_word_position_embedding:84 - input_vector[:3]: [0.027865692973136902, -0.05832936242222786, -0.0853089988231659]
